# v24.1 Standard — Option-wise MC + YNU candidate rerank

GPU notebook. Generates option-wise entailment only for MC validation questions, and uses saved v23 candidates for YNU rerank. Does not train.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

QWEN_MODEL_ID = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
MAX_NEW_TOKENS_OPTION = 96
RUN_OPTIONWISE_MC = True
LIMIT_MC = None  # set e.g. 10 for smoke test

In [ ]:
import os, re, json, glob, math, random, time, subprocess, sys, csv
from pathlib import Path
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

DATA_ROOT = Path("/kaggle/input/datasets/nguyenminhtric/test-pipeline")
V23_ROOT  = Path("/kaggle/input/datasets/yixuanisthebest/v2333333")
OUT_DIR   = Path("/kaggle/working/v24_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAND_PATH = V23_ROOT / "v23_val_candidates.json"
SUMMARY_PATH = V23_ROOT / "v23_val_summary.json"
LORA_DIR = V23_ROOT

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = ["A","B","C","D"]
YNU_LABELS = ["Yes","No","Unknown"]

def norm_answer(x):
    if x is None: return "UNPARSEABLE"
    s = str(x).strip()
    u = s.upper().replace(".", "").replace(":", "")
    if u in {"A","B","C","D"}: return u
    if u in {"YES","Y","TRUE"}: return "Yes"
    if u in {"NO","N","FALSE"}: return "No"
    if u in {"UNKNOWN","UNCERTAIN","UNK","CANNOT BE DETERMINED","NOT ENOUGH INFORMATION"}: return "Unknown"
    m = re.search(r"\b(A|B|C|D|Yes|No|Unknown|Uncertain)\b", s, flags=re.I)
    if m: return norm_answer(m.group(1))
    return "UNPARSEABLE"

def parse_final_answer(text):
    if text is None: return "UNPARSEABLE"
    t = str(text)
    pats = [
        r"Final\s*Answer\s*[:\-]\s*([A-D]|Yes|No|Unknown|Uncertain)\b",
        r"Answer\s*[:\-]\s*([A-D]|Yes|No|Unknown|Uncertain)\b",
    ]
    for p in pats:
        m = re.search(p, t, flags=re.I|re.S)
        if m:
            return norm_answer(m.group(1))
    return norm_answer(t[-100:])

def extract_supporting(text):
    if text is None: return []
    t = str(text)
    nums = []
    m = re.search(r"Supporting\s+Premises\s*[:\-]\s*\[([^\]]*)\]", t, flags=re.I)
    if m:
        nums += [int(x) for x in re.findall(r"\d+", m.group(1))]
    nums += [int(x) for x in re.findall(r"(?:premise|Premise|P)\s*#?\s*(\d+)", t)]
    out = []
    for n in nums:
        if n not in out:
            out.append(n)
    return out

def raw_text(c):
    return str(c.get("raw") or c.get("text") or c.get("output") or c.get("generation") or "")

def cand_answer(c):
    return norm_answer(c.get("answer") or c.get("pred") or parse_final_answer(raw_text(c)))

def cand_support(c):
    sp = c.get("supporting_premises", None)
    if isinstance(sp, list):
        try: return [int(x) for x in sp]
        except Exception: return extract_supporting(str(sp))
    return extract_supporting(raw_text(c))

def is_mc_record(r):
    qt = str(r.get("q_type") or r.get("type") or "").lower()
    gold = norm_answer(r.get("gold") or r.get("answer"))
    q = str(r.get("question",""))
    if gold in MC_LABELS: return True
    if qt == "mc": return True
    if re.search(r"\bA\s*[\.\)]", q) and re.search(r"\bB\s*[\.\)]", q): return True
    return False

def get_gold(r):
    return norm_answer(r.get("gold") or r.get("answer") or r.get("label"))

def metrics_from_preds(golds, preds, title="METRICS"):
    golds = [norm_answer(x) for x in golds]
    preds = [norm_answer(x) for x in preds]
    n = len(golds)
    correct = sum(g==p for g,p in zip(golds,preds))
    rows = {}; f1s = []; weights = []
    print("="*100); print(title); print("="*100)
    print(f"N={n} acc={correct/n if n else 0:.3f}")
    print("-"*100)
    print(f"{'Label':10s} {'P':>8s} {'R':>8s} {'F1':>8s} {'Gold':>8s} {'Pred':>8s}")
    for lab in LABELS:
        tp = sum(g==lab and p==lab for g,p in zip(golds,preds))
        fp = sum(g!=lab and p==lab for g,p in zip(golds,preds))
        fn = sum(g==lab and p!=lab for g,p in zip(golds,preds))
        support = sum(g==lab for g in golds); predc = sum(p==lab for p in preds)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        rows[lab] = dict(precision=prec, recall=rec, f1=f1, support=support, pred_count=predc, tp=tp, fp=fp, fn=fn)
        if support > 0: f1s.append(f1); weights.append(support)
        print(f"{lab:10s} {prec:8.3f} {rec:8.3f} {f1:8.3f} {support:8d} {predc:8d}")
    macro = sum(f1s)/len(f1s) if f1s else 0
    weighted = sum(f*w for f,w in zip(f1s,weights))/sum(weights) if weights else 0
    print("-"*100); print("macro_f1=", round(macro,4), "weighted_f1=", round(weighted,4))
    print("Gold dist:", dict(Counter(golds))); print("Pred dist:", dict(Counter(preds)))
    return {"acc": correct/n if n else 0, "macro_f1": macro, "weighted_f1": weighted, "per_label": rows}

def majority_answer(cands):
    vals = [cand_answer(c) for c in cands]
    vals = [v for v in vals if v != "UNPARSEABLE"]
    if not vals: return "UNPARSEABLE"
    cnt = Counter(vals); order = {lab:i for i,lab in enumerate(["A","B","C","D","Yes","No","Unknown"])}
    return sorted(cnt.items(), key=lambda kv: (-kv[1], order.get(kv[0], 99)))[0][0]

def oracle_answer(r):
    gold = get_gold(r)
    for c in r.get("candidates", []):
        if cand_answer(c) == gold:
            return gold
    return majority_answer(r.get("candidates", []))

def citation_valid_score(c, r=None):
    supp = cand_support(c)
    if not supp: return 0.0
    premises = []
    if isinstance(r, dict):
        premises = r.get("premises") or r.get("premises-NL") or r.get("premises_NL") or []
    if isinstance(premises, list) and len(premises) > 0:
        return 1.0 if all(1 <= int(x) <= len(premises) for x in supp) else -1.0
    return 0.7

def support_len_score(c):
    n = len(cand_support(c))
    if n == 0: return -0.4
    if 1 <= n <= 5: return 0.6
    if 6 <= n <= 8: return 0.1
    return -0.5

def clean_format_score(c):
    ans = cand_answer(c); t = raw_text(c); score = 0.0
    if ans != "UNPARSEABLE": score += 0.5
    if re.search(r"Final\s*Answer\s*[:\-]", t, flags=re.I): score += 0.4
    if re.search(r"Supporting\s+Premises\s*[:\-]", t, flags=re.I): score += 0.4
    return score

def vote_share(answer, cands):
    vals = [cand_answer(c) for c in cands]
    vals = [v for v in vals if v != "UNPARSEABLE"]
    if not vals: return 0.0
    return sum(v == answer for v in vals) / len(vals)

def select_by_score(r, score_fn):
    cands = r.get("candidates", [])
    if not cands: return "UNPARSEABLE", None, -999
    scored = [(score_fn(c, r), i, c) for i,c in enumerate(cands)]
    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)
    s,i,c = scored[0]
    return cand_answer(c), c, s

def to_jsonable(x):
    import numpy as np
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

In [ ]:
assert CAND_PATH.exists(), f"Missing candidates: {CAND_PATH}"
with open(CAND_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Loaded VAL records:", len(results))
print("Sample keys:", sorted(results[0].keys()))
print("Sample candidate keys:", sorted(results[0]["candidates"][0].keys()))
print("Gold dist:", Counter(get_gold(r) for r in results))
print("MC count:", sum(is_mc_record(r) for r in results), "YNU count:", sum(not is_mc_record(r) for r in results))

In [ ]:
def score_ynu_candidate(c, r):
    ans = cand_answer(c)
    cands = r.get("candidates", [])
    score = 0.0
    score += 1.2 * vote_share(ans, cands)
    score += citation_valid_score(c, r)
    score += support_len_score(c)
    score += clean_format_score(c)
    if ans == "Yes": score += 0.8
    if ans == "No": score -= 0.25
    if ans == "Unknown": score -= 0.20
    if ans in MC_LABELS: score -= 2.0
    return score

def ynu_rerank_answer(r):
    return select_by_score(r, score_ynu_candidate)[0]

golds = [get_gold(r) for r in results]
first_preds = [cand_answer(r["candidates"][0]) if r.get("candidates") else "UNPARSEABLE" for r in results]
maj_preds = [majority_answer(r.get("candidates", [])) for r in results]
oracle_preds = [oracle_answer(r) for r in results]
ynu_rerank_preds = [ynu_rerank_answer(r) if not is_mc_record(r) else majority_answer(r.get("candidates", [])) for r in results]

m_first = metrics_from_preds(golds, first_preds, "VAL -- FIRST CANDIDATE")
m_maj = metrics_from_preds(golds, maj_preds, "VAL -- MAJORITY")
m_oracle = metrics_from_preds(golds, oracle_preds, "VAL -- ORACLE AMONG CANDIDATES")
m_ynu = metrics_from_preds(golds, ynu_rerank_preds, "VAL -- YNU RERANK + MC MAJORITY")

In [ ]:
if RUN_OPTIONWISE_MC:
    def _pip(pkg):
        print("Installing", pkg)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages"] + pkg.split(), check=False)
    try:
        import bitsandbytes, peft
    except Exception:
        _pip("bitsandbytes peft accelerate safetensors")
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    torch.cuda.set_device(0)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(LORA_DIR), is_trainable=False)
    model.eval()
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    try:
        model.generation_config.max_length = None
    except Exception:
        pass
    print("Loaded model + LoRA:", LORA_DIR)
    print("LoRA modules:", len([n for n,_ in model.named_modules() if "lora" in n.lower()]))
else:
    model = tokenizer = None

In [ ]:
def get_premises_from_record(r):
    for k in ["premises", "premises-NL", "premises_NL", "premises_nl"]:
        v = r.get(k)
        if isinstance(v, list): return v
    return []

def extract_options_from_question(q):
    q = str(q)
    opts = {}
    matches = list(re.finditer(r"(?m)(?:^|\n|\s)([A-D])\s*[\.\)]\s*", q))
    for idx, m in enumerate(matches):
        lab = m.group(1)
        start = m.end()
        end = matches[idx+1].start() if idx+1 < len(matches) else len(q)
        opts[lab] = q[start:end].strip()
    return opts

def build_option_prompt(r, opt_label, opt_text):
    premises = get_premises_from_record(r)
    prem_txt = "\n".join([f"Premise {i+1}: {p}" for i,p in enumerate(premises)])
    q = str(r.get("question",""))
    return f"""You are a careful logic verifier.

Given the premises, decide whether option {opt_label} is entailed by the premises.

Premises:
{prem_txt}

Question:
{q}

Option {opt_label}:
{opt_text}

Return exactly:
Supported: Yes or No
Evidence: Premise numbers used, if any.
"""

def parse_supported(text):
    t = str(text)
    m = re.search(r"Supported\s*[:\-]\s*(Yes|No)", t, flags=re.I)
    if m: return m.group(1).lower() == "yes"
    if re.search(r"\b(entails|entailed|supported|follows|true)\b", t, flags=re.I):
        if not re.search(r"\bnot\s+(entailed|supported|follow)|does\s+not\s+follow|not\s+true\b", t, flags=re.I):
            return True
    return False

@torch.no_grad()
def generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS_OPTION):
    messages = [{"role":"user", "content": prompt}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

def optionwise_predict_mc(r):
    opts = extract_options_from_question(r.get("question",""))
    if not opts:
        return majority_answer(r.get("candidates", [])), {}
    support, raw = {}, {}
    for lab in MC_LABELS:
        opt_text = opts.get(lab, "")
        if not opt_text:
            support[lab] = False; raw[lab] = ""; continue
        txt = generate_text(build_option_prompt(r, lab, opt_text))
        raw[lab] = txt
        support[lab] = parse_supported(txt)
    supported = [lab for lab,val in support.items() if val]
    if supported:
        vote = Counter(cand_answer(c) for c in r.get("candidates", []))
        supported.sort(key=lambda lab: (vote.get(lab, 0), -MC_LABELS.index(lab)), reverse=True)
        return supported[0], {"support": support, "raw": raw}
    cand_mcs = [cand_answer(c) for c in r.get("candidates", []) if cand_answer(c) in MC_LABELS]
    if cand_mcs:
        return Counter(cand_mcs).most_common(1)[0][0], {"support": support, "raw": raw}
    return majority_answer(r.get("candidates", [])), {"support": support, "raw": raw}

if RUN_OPTIONWISE_MC:
    mc_indices = [i for i,r in enumerate(results) if is_mc_record(r)]
    if LIMIT_MC is not None:
        mc_indices = mc_indices[:LIMIT_MC]
    print("Running option-wise MC on", len(mc_indices), "records")
    option_preds = {}
    option_debug = {}
    t0 = time.time()
    for k,i in enumerate(mc_indices, 1):
        pred, dbg = optionwise_predict_mc(results[i])
        option_preds[i] = pred
        option_debug[i] = dbg
        if k % 10 == 0 or k == len(mc_indices):
            print(f"  {k}/{len(mc_indices)} done in {(time.time()-t0)/60:.1f} min")
    with open(OUT_DIR/"v24_1_optionwise_mc_debug.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(option_debug), f, ensure_ascii=False, indent=2)
else:
    option_preds = {}

In [ ]:
final_preds = []
for i,r in enumerate(results):
    if is_mc_record(r):
        final_preds.append(option_preds.get(i, majority_answer(r.get("candidates", []))))
    else:
        final_preds.append(ynu_rerank_answer(r))

m_final = metrics_from_preds(golds, final_preds, "VAL -- v24.1 STANDARD: MC OPTIONWISE + YNU RERANK")

summary = {
    "first": m_first,
    "majority": m_maj,
    "oracle": m_oracle,
    "ynu_rerank_mc_majority": m_ynu,
    "v24_1_standard": m_final,
    "option_preds_count": len(option_preds),
}
with open(OUT_DIR/"v24_1_standard_summary.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(summary), f, ensure_ascii=False, indent=2)
print("Saved:", OUT_DIR/"v24_1_standard_summary.json")